In [1]:
%load_ext autoreload
%autoreload 2
import json
import pandas as pd

from dfir_copilot.agent.graph import build_agent
from dfir_copilot.agent.llm import make_llm
from dfir_copilot.engine.query_engine import QueryEngine
from dfir_copilot.evidence.ledger import Ledger

engine = QueryEngine("/workspace/data/processed/three_months.parquet", timeout_s=120)
ledger = Ledger.open("IDOR-INVOICES-2020Q4", engine, analyst="Eder")
agent = build_agent(engine, ledger, make_llm(), max_steps=12)
n0 = len(ledger.entries())   # marca de inicio, para ver después qué hizo el agente
print("modelo:", agent.model_name, "| entradas del ledger:", n0)

modelo: claude-sonnet-5-5 | entradas del ledger: 110


In [4]:
r = agent.ask("¿Hay actividad anómala sobre /invoices/search? Investiga el dataset, formula hipótesis y pruébalas.")
print(r.status, "| pasos:", r.steps, "| tokens:", r.tokens)
print(r.answer or "(pausado: hay hipótesis pendientes de tu aprobación)")

AgentBusy: Hay una aprobación pendiente en este hilo: llama a resume() o resolve() primero

In [3]:
for e in ledger.entries()[n0:]:
    d = e["data"]
    if e["type"] == "tool_call":
        print(f"{e['seq']:>3} herramienta  {d['tool']:<17} {json.dumps(d['args'], ensure_ascii=False)[:110]}  {'ok' if d['ok'] else 'ERROR'}")
    elif e["type"] == "hypothesis":
        print(f"{e['seq']:>3} hipótesis    {d['hypothesis_id']}  {d['statement'][:100]}")
    elif e["type"] == "hypothesis_update":
        print(f"{e['seq']:>3} estado       {d['hypothesis_id']}  {d['from']} -> {d['to']}  ({d['decision']})")
    elif e["type"] == "agent_turn":
        print(f"{e['seq']:>3} turno        {d['status']}  pasos={d['steps']}  tokens={d['tokens']}  {d.get('error') or ''}")

112 herramienta  describe_dataset  {}  ok
133 herramienta  run_detectors     {}  ok
135 herramienta  run_query         {"sql": "SELECT user_id, status_code, count(*) n, count(distinct x_invoice_id) ids, min(timestamp_utc) t0, max  ok
137 herramienta  run_query         {"sql": "SELECT CASE WHEN user_id IN ('jacintobuyer','mantecoluser','marcianobuyer','mateuser') THEN 'grupo' E  ok
139 herramienta  run_query         {"sql": "SELECT date_trunc('month', timestamp_utc) m, CASE WHEN user_id IN ('jacintobuyer','mantecoluser','mar  ok
141 herramienta  run_query         {"sql": "SELECT user_id, src_ip, count(*) n, min(timestamp_utc) t0, max(timestamp_utc) t1, count(distinct x_in  ok
143 herramienta  run_query         {"sql": "WITH base AS (SELECT DISTINCT x_invoice_id FROM logs WHERE user_id NOT IN ('jacintobuyer','mantecolus  ok
145 herramienta  run_query         {"sql": "SELECT http_method, endpoint, x_token_type, count(*) n, count(distinct query_string) q FROM logs WHER  ok
147 herramienta 

In [5]:
for req in agent.pending():
    print(f"{req['hypothesis_id']}: {req['estado_actual']} -> {req['estado_solicitado']}")
    print("  Hipótesis:    ", req["hipotesis"])
    print("  Justificación:", req["justificacion"])
    for ev in req["evidencia"]:
        print(f"  [{ev['tipo']}] {ev['ref']}: {ev['detalle'][:160]}")

h-60d4f260: en_prueba -> confirmada
  Hipótesis:     Cuatro cuentas (jacintobuyer, mantecoluser, marcianobuyer, mateuser) coordinadas desde 5 IPs exclusivas enumeran x_invoice_id en /invoices/search con herramientas automatizadas, con escalada en el último tercio del periodo.
  Justificación: 4 cuentas, 5 IPs exclusivas (cluster cerrado), 100% UA automatizados (Scrapy/crawler4j/wget), 7.1-7.2k invoice_id distintos vs 5001 de los pares, ~3.3k IDs fuera del conjunto común, y rampa 27-32x entre primer y último tercio. Matiz: volumen absoluto bajo (~15k/cuenta vs ~143k de pares). Pendiente d…[+26 car.]
  [hallazgo] f-725c25c75086: [high] 4 identidad(es) operan en exclusiva desde 5 IP(s): 12392 peticiones por IP frente a 25 típico por actor.
  [hallazgo] f-ae1bfb870c04: [medium] mateuser: el 100% de sus peticiones (15569 de 15569) viene de clientes automatizados: Scrapy/2.3.0 (+https://scrapy.org), crawler4j, wget.
  [hallazgo] f-829507c4430d: [medium] jacintobuyer accedió a 7200 x_invoice_